# **W2V - Simple Implementation**

<img src=https://miro.medium.com/v2/resize:fit:720/format:webp/1*bBETsVNLyjnaFJgM9avkeQ.png></img>






**W2V Examples:**

In [1]:
!python -m spacy download en_core_web_lg

/Users/hd/Desktop/Machine Learning/.venv/bin/python: No module named spacy


In [2]:
import spacy
#getting the vector representations for words with spacy
nlp = spacy.load("en_core_web_lg")
words = ["dog","cat","house","car","truck"]
vecs = []
for word in words:
  vecs.append(nlp(word).vector)

ModuleNotFoundError: No module named 'spacy'

In [ ]:
#using cosine similarity instead of euclidean distance
from sklearn.metrics.pairwise import cosine_similarity

print (f"Similarity: {words[0]} - {words[1]}: {cosine_similarity([vecs[0]],[vecs[1]])}")
print (f"Similarity: {words[0]} - {words[2]}: {cosine_similarity([vecs[0]],[vecs[2]])}")
print (f"Similarity: {words[0]} - {words[3]}: {cosine_similarity([vecs[0]],[vecs[3]])}")
print (f"Similarity: {words[0]} - {words[4]}: {cosine_similarity([vecs[0]],[vecs[4]])}")
print (f"Similarity: {words[3]} - {words[4]}: {cosine_similarity([vecs[3]],[vecs[4]])}")

# **Custom implementation**

In [ ]:
import torch
import torch.nn as nn

class w2v(torch.nn.Module):
  def __init__(self, num_embeddings, embedding_dim):
    super(w2v, self).__init__()
    self.embeddings = nn.Parameter(torch.rand((num_embeddings,embedding_dim),requires_grad=True))
    self.classifier = nn.Linear(embedding_dim,num_embeddings)

  def forward(self,ids):
    words = self.embeddings[ids]
    classifierIn = torch.sum(words,dim=0)
    out = self.classifier(classifierIn)
    return out

  def getEmbedding(self,id):
    id = torch.LongTensor([id])
    return self.embeddings[id]


In [ ]:
corpus = ["the dog is nice","the cat is nice"]

def toId(texts):
  idDict = {}
  nrWords = 0
  for text in texts:
    for word in text.split():
      if not word in idDict:
        idDict[word] = nrWords
        nrWords += 1
  return idDict

idDict = toId(corpus)
print (idDict)


{'the': 0, 'dog': 1, 'is': 2, 'nice': 3, 'cat': 4}


In [ ]:
def applyAndTransform(idDict,corpus):
  corpusAsIds = []
  for text in corpus:
    textAsNrs = []
    for word in text.split():
      id = idDict[word]
      textAsNrs.append(id)
    corpusAsIds.append(textAsNrs)

  return torch.LongTensor(corpusAsIds)

applyAndTransform(idDict,corpus)

tensor([[0, 1, 2, 3],
        [0, 4, 2, 3]])

In [ ]:
inputs = torch.LongTensor([[0,2],[1,3],[0,2],[4,3]])
targets = torch.LongTensor([[1],[2],[4],[2]])

In [ ]:
model = w2v(5,10)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for _ in range(50):
  lossAbs = 0
  for sample,target in zip(inputs,targets):
    outputs = model(sample)
    # Add a batch dimension to outputs to match CrossEntropyLoss expectation
    loss = criterion(outputs.unsqueeze(0),target)
    # Backward and optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    lossAbs += float(loss)

  print (lossAbs/inputs.shape[0])

print("Training finished.")

2.563458800315857
2.5014111399650574
2.4435611963272095
2.3873379230499268
2.332662522792816
2.279547870159149
2.2280110716819763
2.178059548139572
2.1296889781951904
2.082883834838867
2.0376195907592773
1.9938653409481049
1.9515862464904785
1.910744845867157
1.8713031113147736
1.8332227170467377
1.7964653670787811
1.7609933018684387
1.72676882147789
1.6937546133995056
1.6619137227535248
1.631209373474121
1.601605385541916
1.5730660557746887
1.5455556809902191
1.5190400779247284
1.4934846758842468
1.4688560962677002
1.4451214969158173
1.4222485721111298
1.4002056419849396
1.3789620697498322
1.358487606048584
1.3387527763843536
1.3197287619113922
1.3013874143362045
1.2837015986442566
1.2666444033384323
1.2501901686191559
1.2343134880065918
1.2189900130033493
1.2041959762573242
1.1899083107709885
1.1761046648025513
1.1627635210752487
1.149864062666893
1.1373861134052277
1.1253103017807007
1.1136180013418198
1.1022910922765732
Training finished.
